# บท 02 · Python ที่รันซ้ำได้

ข้อมูลทั้งหมดสร้างขึ้นเพื่อการเรียนรู้ ไม่มีการเชื่อม API หรือใช้ข้อมูลตลาดจริง เริ่มจากราคาปิดห้าค่า สร้างฟังก์ชัน SMA แล้วตรวจผลด้วยนิยามโดยตรง

**วิธีใช้:** รันเซลล์จากบนลงล่างใน kernel ใหม่ ใช้ Python 3.12 พร้อม NumPy 2.5.3 และ pandas 2.3.2 ที่ตรวจไว้ ผลลัพธ์ถูกบันทึกจากการรันเซลล์จริงในกระบวนการ Python ใหม่


## 1. ตรวจสภาพแวดล้อม

บันทึกเวอร์ชันพร้อมข้อมูลและพารามิเตอร์ เพื่ออธิบายได้ว่าเหตุใดผลต่างจากการทดลองก่อนหน้า


In [1]:
import platform
import numpy as np
import pandas as pd
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__})


{'python': '3.12.14', 'numpy': '2.5.3', 'pandas': '2.3.2'}


## 2. สร้างข้อมูลในตัว

ใช้เลขแท่งแทนวันที่ ราคามีหน่วยสมมติ USD ไม่มีหุ้นหรือปฏิทินตลาดจริง


In [2]:
prices = [100.0, 102.0, 101.0, 104.0, 103.0]
close = pd.Series(prices, index=pd.RangeIndex(1, 6, name="bar"), name="close")
data = close.to_frame()
print(data.to_string())
print("First position / first label:", close.iloc[0], close.loc[1])


     close
bar       
1    100.0
2    102.0
3    101.0
4    104.0
5    103.0
First position / first label: 100.0 100.0


## 3. ฟังก์ชันคำนวณที่ไม่แก้ input

ฟังก์ชันปฏิเสธหน้าต่างที่ใช้ไม่ได้ ราคาที่ไม่เป็นบวกหรือไม่จำกัด และดัชนีที่ซ้ำหรือย้อนลำดับ


In [3]:
def trailing_mean(close, window=3):
    """Return a full-window trailing mean without modifying the input."""
    if isinstance(window, bool) or not isinstance(window, int) or window < 1:
        raise ValueError("window must be a positive integer")
    values = pd.Series(close, dtype="float64", copy=True)
    if not np.isfinite(values).all() or (values <= 0).any():
        raise ValueError("prices must be finite and positive")
    if not values.index.is_monotonic_increasing or not values.index.is_unique:
        raise ValueError("index must be ordered and unique")
    return values.rolling(window, min_periods=window).mean()
print("Function ready: trailing_mean")


Function ready: trailing_mean


## 4. คำนวณ SMA 3

สองแท่งแรกยังไม่ครบหน้าต่าง จึงต้องเป็น NaN ค่าแถวสุดท้ายตรวจได้ด้วย (101 + 104 + 103) / 3


In [4]:
data["sma3"] = trailing_mean(close, 3)
assert data["sma3"].iloc[:2].isna().all()
assert np.isclose(data["sma3"].iloc[-1], 308 / 3)
print(data.round(6).to_string())


     close        sma3
bar                   
1    100.0         NaN
2    102.0         NaN
3    101.0  101.000000
4    104.0  102.333333
5    103.0  102.666667


## 5. ผลตอบแทนธรรมดาต้องทบด้วยผลคูณ

pct_change คืนสัดส่วน ไม่ใช่ค่าที่คูณ 100 แล้ว ผล 3% เป็นการเปลี่ยนแปลงของราคาจำลองเท่านั้น ยังไม่ใช่ผลตอบแทนกลยุทธ์


In [5]:
data["return"] = close.pct_change(fill_method=None)
total = (1 + data["return"].dropna()).prod() - 1
assert np.isclose(total, close.iloc[-1] / close.iloc[0] - 1)
print(data.round(6).to_string())
print(f"Compounded: {total:.6%}; arithmetic sum: {data['return'].sum():.6%}")


     close        sma3    return
bar                             
1    100.0         NaN       NaN
2    102.0         NaN  0.020000
3    101.0  101.000000 -0.009804
4    104.0  102.333333  0.029703
5    103.0  102.666667 -0.009615
Compounded: 3.000000%; arithmetic sum: 3.028366%


## 6. ตรวจ input และการไม่เปลี่ยนข้อมูลต้นฉบับ

ข้อผิดพลาดที่ตั้งใจปฏิเสธควรเกิดก่อนการสร้างกราฟหรือสัญญาณ เราทดสอบกรณีราคาศูนย์และหน้าต่างไม่ถูกต้อง


In [6]:
original = close.copy(deep=True)
_ = trailing_mean(close, 3)
pd.testing.assert_series_equal(close, original)
rejected = 0
for bad_prices, bad_window in [([100, 0, 101], 3), ([100, 101], 0), ([100, 101], True)]:
    try:
        trailing_mean(bad_prices, bad_window)
    except ValueError as error:
        rejected += 1
        print("Rejected:", str(error))
assert rejected == 3
print("Input unchanged; all 3 invalid cases rejected.")


Rejected: prices must be finite and positive
Rejected: window must be a positive integer
Rejected: window must be a positive integer
Input unchanged; all 3 invalid cases rejected.


## 7. บันทึกข้อมูลกำกับและ hash

SHA-256 ใช้เปรียบเทียบข้อมูลภายใต้ serialization เดียวกัน ไม่รับรองความถูกต้องของข้อมูลและไม่พิสูจน์ว่าเป็นข้อมูลจากตลาด


In [7]:
import hashlib
import json
payload = json.dumps(prices, separators=(",", ":")).encode()
checksum = hashlib.sha256(payload).hexdigest()
assert checksum == "272ac23eed5c4834bbd489be71dfd88fe1d32fa8df6f38868ba068ceb1ab20aa"
print("Input SHA256:", checksum)
print(json.dumps({"symbol": "SYNTHETIC", "calendar": None, "currency": "illustrative USD", "window": 3, "rows": len(prices)}, indent=2))


Input SHA256: 272ac23eed5c4834bbd489be71dfd88fe1d32fa8df6f38868ba068ceb1ab20aa
{
  "symbol": "SYNTHETIC",
  "calendar": null,
  "currency": "illustrative USD",
  "window": 3,
  "rows": 5
}


## 8. แบบฝึกหัดและเฉลย

เปลี่ยนราคาปิดสุดท้ายเป็น 105 ก่อนรัน ลองคำนวณ SMA สุดท้ายและผลตอบแทนด้วยมือ ผล SMA ก่อนแท่งสุดท้ายต้องคงเดิม


In [8]:
changed = close.copy()
changed.iloc[-1] = 105
new_sma = trailing_mean(changed, 3)
new_return = changed.iloc[-1] / changed.iloc[0] - 1
pd.testing.assert_series_equal(new_sma.iloc[:-1], data["sma3"].iloc[:-1], check_names=False)
assert np.isclose(new_sma.iloc[-1], 310 / 3)
assert np.isclose(new_return, .05)
assert trailing_mean(close, 6).isna().all()
print(f"New SMA3: {new_sma.iloc[-1]:.6f}; price change: {new_return:.6%}")
print("Prior SMA values unchanged; SMA6 has no complete window.")


New SMA3: 103.333333; price change: 5.000000%
Prior SMA values unchanged; SMA6 has no complete window.


## แหล่งอ้างอิง

- [Python 3.12 venv](https://docs.python.org/3.12/library/venv.html)
- [pandas 2.3 rolling mean](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.core.window.rolling.Rolling.mean.html)
- [pandas 2.3 pct_change](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.Series.pct_change.html)
- Hilpisch, Python for Algorithmic Trading, บท 2 หน้าเล่ม 17–18, 27 (PDF 37–38, 47): แนวคิด environment; โค้ดและข้อความนี้เขียนใหม่

ตรวจเอกสาร 11 กันยายน 2026
